# The Formal State-Machine Algorithm for Iterative DFS

To correctly simulate recursion, we must explicitly manage the state transitions that the call stack manages implicitly. The key insight is that a recursive call `DFS-VISIT(u)` remains active (and `u` remains **GRAY**) from the moment it is called until all its children's recursive calls have returned. Our stack must model this persistence.

The most faithful simulation treats the stack as a path of discovery. A vertex is only popped and "finished" after it has been confirmed that all its discovery paths have been exhausted.

---

## Algorithm Components

- **States (Colors):**  
    We use the same `WHITE`, `GRAY`, `BLACK` scheme.
    - **WHITE:** Undiscovered.
    - **GRAY:** Discovered, on the stack, and currently being explored. Corresponds to being on the recursion call stack.
    - **BLACK:** Finished; all descendants have been explored.

- **Data Structures:**  
    An explicit stack `S`. The main DFS driver loop remains the same to handle disconnected components.

---

## The Correct Iterative DFS-VISIT

This algorithm perfectly mirrors the recursive flow:  
- `peek()` at the top of the stack to see where we are,  
- find a new path (a `WHITE` neighbor),  
- or backtrack if there are no new paths.

```pseudo
// CORRECT-ITERATIVE-DFS-VISIT(Graph G, Vertex s)
1.  S = new Stack()
2.  S.push(s)

3.  while S is not empty:
4.      u = S.peek() // Look at the current vertex without removing it

5.      // First time we see u at the top of the stack
6.      if u.color == WHITE:
7.          time = time + 1
8.          u.d = time
9.          u.color = GRAY
10.         u.pi = (S.size() > 1) ? S.second_from_top() : NIL // Get parent from stack

11.     // Find the next unexplored path from u
12.     found_path = false
13.     for each vertex v in G.Adj[u]:
14.         if v.color == WHITE:
15.             // Found a new path to explore. Push v and loop.
16.             S.push(v)
17.             found_path = true
18.             break // Exit neighbor loop to process v from top of stack

19.     // If no new path was found from u, its exploration is complete.
20.     if not found_path:
21.         // Backtrack: u is finished.
22.
```

In [1]:
from __future__ import annotations
import enum
from dataclasses import dataclass, field
from typing import List, Dict, Optional, TypeAlias

# --- Type and State Definitions ---

VertexID: TypeAlias = str | int

class Color(enum.Enum):
    """Represents the three states of a vertex during DFS."""
    WHITE = 0  # Undiscovered
    GRAY = 1   # Discovered, on the stack, descendants being explored
    BLACK = 2  # Finished, all descendants fully explored

@dataclass
class Vertex:
    """A dataclass to encapsulate all state associated with a vertex."""
    id: VertexID
    color: Color = Color.WHITE
    d: int = -1  # Discovery time
    f: int = -1  # Finish time
    pi: Optional[Vertex] = None
    
    # Adjacency list is part of the graph structure, not the vertex state itself.
    # We will manage it in the Graph class for a cleaner design.
    
    def __repr__(self) -> str:
        pi_id = self.pi.id if self.pi else "NIL"
        return (f"Vertex(id={self.id}, color={self.color.name}, "
                f"d={self.d}, f={self.f}, pi={pi_id})")

class Graph:
    """Represents the graph using an adjacency list model."""
    def __init__(self):
        self._vertices: Dict[VertexID, Vertex] = {}
        self._adj: Dict[VertexID, List[Vertex]] = {}

    def add_vertex(self, vertex_id: VertexID):
        if vertex_id not in self._vertices:
            self._vertices[vertex_id] = Vertex(id=vertex_id)
            self._adj[vertex_id] = []

    def add_edge(self, u_id: VertexID, v_id: VertexID):
        self.add_vertex(u_id)
        self.add_vertex(v_id)
        self._adj[u_id].append(self._vertices[v_id])
        
    @property
    def vertices(self) -> List[Vertex]:
        return list(self._vertices.values())

    def __getitem__(self, vertex_id: VertexID) -> Vertex:
        return self._vertices[vertex_id]

    def adj(self, u: Vertex) -> List[Vertex]:
        return self._adj[u.id]

# --- The Iterative DFS Algorithm ---

class DFSRunner:
    """Encapsulates the logic and state for a complete DFS run."""
    def __init__(self, graph: Graph):
        self._graph = graph
        self._time = 0

    def run_dfs(self):
        """Main driver loop to run DFS on the entire graph."""
        for vertex in self._graph.vertices:
            if vertex.color == Color.WHITE:
                self._iterative_dfs_visit(vertex)

    def _iterative_dfs_visit(self, start_vertex: Vertex):
        """
        Performs a semantically equivalent iterative DFS from a start vertex,
        perfectly simulating the recursive call stack and preserving all formal
        properties of DFS (discovery/finish times, parent pointers).
        """
        stack: List[Vertex] = [start_vertex]

        while stack:
            u = stack[-1]  # Peek at the vertex on top of the stack

            # State: DISCOVER
            # This block corresponds to the top of the recursive DFS_VISIT function.
            # It runs only the first time a vertex is at the top of the stack.
            if u.color == Color.WHITE:
                self._time += 1
                u.d = self._time
                u.color = Color.GRAY
            
            # State: EXPLORE
            # This block corresponds to the for-loop in the recursive version.
            # We search for an undiscovered neighbor to deepen the search.
            # The `for...else` construct elegantly handles the logic without a flag.
            for v in self._graph.adj(u):
                if v.color == Color.WHITE:
                    v.pi = u  # Set parent before pushing, simplifying logic
                    stack.append(v)
                    break  # Found a new path, restart while loop with v on top
            else:
                # State: FINISH
                # The `else` block executes only if the `for` loop completed
                # without a `break`. This means all of u's neighbors have been
                # visited. We can now finish u, simulating the return of a
                # recursive call.
                u = stack.pop()
                u.color = Color.BLACK
                self._time += 1
                u.f = self._time

In [2]:
# --- Verification with an Example ---
if __name__ == "__main__":
    # Create the graph from the lecture (nodes 1-8)
    g = Graph()
    edges = [
        (1, 4), (2, 1), (2, 3), (3, 1), (4, 3), (4, 5),
        (4, 6), (5, 6), (7, 8), (8, 7)
    ]
    all_nodes = {1, 2, 3, 4, 5, 6, 7, 8}
    
    for node_id in all_nodes:
        g.add_vertex(node_id)
    for u, v in edges:
        g.add_edge(u, v)

    # Run the DFS
    runner = DFSRunner(g)
    runner.run_dfs()

    # Print results in a clear, tabular format for verification
    print(f"{'Vertex':<10}{'Color':<10}{'d':<5}{'f':<5}{'Parent':<10}")
    print("-" * 40)
    # Sort vertices by ID for consistent output
    sorted_vertices = sorted(g.vertices, key=lambda v: v.id)
    for v in sorted_vertices:
        parent_id = v.pi.id if v.pi else "NIL"
        print(f"{v.id:<10}{v.color.name:<10}{v.d:<5}{v.f:<5}{parent_id:<10}")

Vertex    Color     d    f    Parent    
----------------------------------------
1         BLACK     1    10   NIL       
2         BLACK     11   12   NIL       
3         BLACK     3    4    4         
4         BLACK     2    9    1         
5         BLACK     5    8    4         
6         BLACK     6    7    5         
7         BLACK     13   16   NIL       
8         BLACK     14   15   7         


In [5]:
example_graph = {"s": ["a", "c"],
             "a": ["s", "b", "d"],
             "b": ["a", "e"],
             "c": ["s", "d", "f"],
             "d": ["a", "c", "e", "g"],
             "e": ["b", "d", "h"],
             "f": ["c", "g"],
             "g": ["d", "f", "h"],
             "h": ["e", "g"]}

G = Graph()
for u, vs in example_graph.items():
    G.add_vertex(u)
    for v in vs:
        G.add_edge(u, v)
runner = DFSRunner(G)
runner.run_dfs()
# Print results in a clear, tabular format for verification
print(f"{'Vertex':<10}{'Color':<10}{'d':<5}{'f':<5}{'Parent':<10}")
print("-" * 40)
sorted_vertices = sorted(G.vertices, key=lambda v: v.f, reverse=True)
for v in sorted_vertices:
    parent_id = v.pi.id if v.pi else "NIL"
    print(f"{v.id:<10}{v.color.name:<10}{v.d:<5}{v.f:<5}{parent_id:<10}")


Vertex    Color     d    f    Parent    
----------------------------------------
s         BLACK     1    18   NIL       
a         BLACK     2    17   s         
b         BLACK     3    16   a         
e         BLACK     4    15   b         
d         BLACK     5    14   e         
c         BLACK     6    13   d         
f         BLACK     7    12   c         
g         BLACK     8    11   f         
h         BLACK     9    10   g         
